In [ ]:
import time

import numpy as np
import scipy.sparse as sp
import scanpy as sc
import lamindb as ln

In [ ]:
# Loads the QC-filtered combined object (same artifact as 03_processing); read-only, nothing is saved.
loaded = ln.Artifact.get("ykkwAQiNc66lfZ340001").load()

In [ ]:
# 1. What is .X as loaded, before any processing?
{"type": type(loaded.X).__name__, "dtype": loaded.X.dtype, "shape": loaded.shape}

In [ ]:
# Makes two in-memory copies of the loaded object; neither touches disk.
dense_path = loaded.copy()
sparse_path = loaded.copy()

# Memory only: forces .X in dense_path to a dense ndarray (no-op if it already is one).
dense_path.X = dense_path.X.toarray() if sp.issparse(dense_path.X) else dense_path.X

In [ ]:
# Memory only: rescales .X of both copies to the median total counts (default target_sum).
sc.pp.normalize_total(dense_path, inplace=True)
sc.pp.normalize_total(sparse_path, inplace=True)

# 2. Does normalize_total preserve the storage type?
{"dense_path": type(dense_path.X).__name__, "sparse_path": type(sparse_path.X).__name__}

In [ ]:
# Memory only: the line under test, applied to sparse_path.X; values are not changed.
sparse_path.X = sp.csr_matrix(sparse_path.X)

# 3. Density and memory: dense array vs CSR (data + indices + indptr), in MB.
dense_array = np.asarray(dense_path.X)
csr = sparse_path.X
{
    "density": csr.nnz / (csr.shape[0] * csr.shape[1]),
    "dense_MB": dense_array.nbytes / 1e6,
    "csr_MB": (csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes) / 1e6,
}

In [ ]:
# 4. Are the values identical after conversion?
np.array_equal(dense_array, csr.toarray())

In [ ]:
# Memory only: log1p then scale on each copy; .X is overwritten, timings are collected per path.
timings = {}

start = time.perf_counter()
sc.pp.log1p(dense_path)
sc.pp.scale(dense_path, zero_center=False, max_value=10)
timings["dense_s"] = time.perf_counter() - start

start = time.perf_counter()
sc.pp.log1p(sparse_path)
sc.pp.scale(sparse_path, zero_center=False, max_value=10)
timings["sparse_s"] = time.perf_counter() - start

# 5. Time for log1p + scale: dense vs sparse; also whether the scaled values agree.
timings["max_abs_diff"] = np.abs(np.asarray(dense_path.X) - sparse_path.X.toarray()).max()
timings

In [ ]:
# Memory only: adds obsm['X_pca'] to each copy (30 comps, same seed).
sc.pp.pca(dense_path, n_comps=30, random_state=0)
sc.pp.pca(sparse_path, n_comps=30, random_state=0)

# 6. Do the two paths give the same PCA? Sign flips per component are allowed, so compare absolute values.
np.abs(np.abs(dense_path.obsm["X_pca"]) - np.abs(sparse_path.obsm["X_pca"])).max()